In [1]:
from pathlib import Path
import sys

import pandas as pd
import numpy as np

PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.append(str(PROJECT_ROOT))

print(PROJECT_ROOT)

c:\Users\keebs\OneDrive\Documentos\INTELIGENCIA ARTIFICIAL\MATERIAS 5S\PRACTICA PROFESIONALIZANTE 2\pf\bekantor-demand-intelligence


In [2]:
from src.data.load_data import load_raw_data
from src.data.clean_data import minimal_cleaning
from src.data.panel_diagnostics import (
    add_store_operation_flags,
    add_zero_taxonomy,
)

datasets = load_raw_data()
datasets = minimal_cleaning(datasets)

sales = datasets["sales"]

sales, apertura = add_store_operation_flags(sales)
sales = add_zero_taxonomy(sales)

sales_modelable = sales[
    sales["tienda_operativa"].eq(1)
    & sales["familia_surtida"].eq(1)
].copy()

print(f"Panel modelable: {len(sales_modelable):,}")
print(
    sales_modelable["date"].min().date(),
    "→",
    sales_modelable["date"].max().date(),
)

Panel modelable: 2,697,207
2013-01-01 → 2017-08-15


In [3]:
from src.models.validation import temporal_train_valid_split

train, validation, validation_start, validation_end = (
    temporal_train_valid_split(
        sales_modelable,
        validation_days=15,
    )
)

print("TRAIN")
print(train["date"].min().date(), "→", train["date"].max().date())
print(f"Registros: {len(train):,}")

print("\nVALIDACIÓN")
print(validation_start.date(), "→", validation_end.date())
print(f"Registros: {len(validation):,}")
print(f"Días: {validation['date'].nunique()}")

TRAIN
2013-01-01 → 2017-07-31
Registros: 2,671,272

VALIDACIÓN
2017-08-01 → 2017-08-15
Registros: 25,935
Días: 15


In [4]:
assert train["date"].max() < validation["date"].min()
assert validation["date"].nunique() == 15
assert len(train) + len(validation) == len(sales_modelable)

print("✓ Split temporal correcto")
print("✓ Sin solapamiento")
print("✓ 15 días de validación")

✓ Split temporal correcto
✓ Sin solapamiento
✓ 15 días de validación


In [5]:
from src.features.model_features import add_known_features

features_base = add_known_features(
    sales_modelable=sales_modelable,
    sales=sales,
    stores=datasets["stores"],
    holidays=datasets["holidays"],
)

print(f"Filas originales: {len(sales_modelable):,}")
print(f"Filas con features: {len(features_base):,}")

features_base.head()

Filas originales: 2,697,207
Filas con features: 2,697,207


,id,date,store_nbr,family,sales,onpromotion,tienda_operativa,familia_surtida,tipo_cero,year,...,type,cluster,is_holiday,is_event,is_transfer,is_additional,is_bridge,is_work_day,is_new_year_closure,event_count
0,581,2013-01-01,25,LAWN AND GARDEN,2.0,0,True,True,Venta positiva,2013,...,D,1,1,0,0,0,0,0,1,1
1,579,2013-01-01,25,HOME CARE,0.0,0,True,True,Cero real de demanda,2013,...,D,1,1,0,0,0,0,0,1,1
2,578,2013-01-01,25,HOME APPLIANCES,0.0,0,True,True,Cero real de demanda,2013,...,D,1,1,0,0,0,0,0,1,1
3,577,2013-01-01,25,HOME AND KITCHEN II,0.0,0,True,True,Cero real de demanda,2013,...,D,1,1,0,0,0,0,0,1,1
4,576,2013-01-01,25,HOME AND KITCHEN I,0.0,0,True,True,Cero real de demanda,2013,...,D,1,1,0,0,0,0,0,1,1


In [6]:
assert len(features_base) == len(sales_modelable)

assert not features_base[
    ["city", "state", "type", "cluster"]
].isna().any().any()

assert not features_base[
    [
        "is_holiday",
        "is_event",
        "is_transfer",
        "is_additional",
        "is_bridge",
        "is_work_day",
        "is_new_year_closure",
    ]
].isna().any().any()

print("✓ Cantidad de filas preservada")
print("✓ Metadata de tiendas completa")
print("✓ Calendario completo")
print("✓ Sin utilizar ventas futuras")

✓ Cantidad de filas preservada
✓ Metadata de tiendas completa
✓ Calendario completo
✓ Sin utilizar ventas futuras


In [7]:
features_base.columns.tolist()

['id',
 'date',
 'store_nbr',
 'family',
 'sales',
 'onpromotion',
 'tienda_operativa',
 'familia_surtida',
 'tipo_cero',
 'year',
 'month',
 'day',
 'day_of_week',
 'week_of_year',
 'is_weekend',
 'city',
 'state',
 'type',
 'cluster',
 'is_holiday',
 'is_event',
 'is_transfer',
 'is_additional',
 'is_bridge',
 'is_work_day',
 'is_new_year_closure',
 'event_count']

In [8]:
from src.features.model_features import add_lag_features

features_lags = add_lag_features(
    features_base,
    lags=(1, 7, 14, 28),
)

features_lags[
    [
        "date",
        "store_nbr",
        "family",
        "sales",
        "lag_1",
        "lag_7",
        "lag_14",
        "lag_28",
    ]
].head(40)

,date,store_nbr,family,sales,lag_1,lag_7,lag_14,lag_28
0,2013-01-01,25,LAWN AND GARDEN,2.000000,NaN,NaN,NaN,NaN
1,2013-01-01,25,HOME CARE,0.000000,NaN,NaN,NaN,NaN
2,2013-01-01,25,HOME APPLIANCES,0.000000,NaN,NaN,NaN,NaN
3,2013-01-01,25,HOME AND KITCHEN II,0.000000,NaN,NaN,NaN,NaN
4,2013-01-01,25,HOME AND KITCHEN I,0.000000,NaN,NaN,NaN,NaN
5,2013-01-01,25,HARDWARE,0.000000,NaN,NaN,NaN,NaN
6,2013-01-01,25,GROCERY II,15.000000,NaN,NaN,NaN,NaN
7,2013-01-01,25,GROCERY I,700.000000,NaN,NaN,NaN,NaN
8,2013-01-01,25,FROZEN FOODS,29.654999,NaN,NaN,NaN,NaN
9,2013-01-01,25,EGGS,46.000000,NaN,NaN,NaN,NaN


In [9]:
assert len(features_lags) == len(features_base)

for lag in [1, 7, 14, 28]:
    assert f"lag_{lag}" in features_lags.columns

print("✓ Filas preservadas")
print("✓ lag_1 creado")
print("✓ lag_7 creado")
print("✓ lag_14 creado")
print("✓ lag_28 creado")

features_lags[
    ["lag_1", "lag_7", "lag_14", "lag_28"]
].isna().mean()

✓ Filas preservadas
✓ lag_1 creado
✓ lag_7 creado
✓ lag_14 creado
✓ lag_28 creado


lag_1     0.003028
lag_7     0.006875
lag_14    0.011362
lag_28    0.020336
dtype: float64

In [10]:
from src.features.model_features import add_rolling_features

features_full = add_rolling_features(
    features_lags,
    windows=(7, 14, 28),
)

features_full[
    [
        "date",
        "store_nbr",
        "family",
        "sales",
        "lag_7",
        "rolling_mean_7",
        "rolling_mean_14",
        "rolling_mean_28",
    ]
].head(40)

,date,store_nbr,family,sales,lag_7,rolling_mean_7,rolling_mean_14,rolling_mean_28
0,2013-01-01,25,LAWN AND GARDEN,2.000000,NaN,NaN,NaN,NaN
1,2013-01-01,25,HOME CARE,0.000000,NaN,NaN,NaN,NaN
2,2013-01-01,25,HOME APPLIANCES,0.000000,NaN,NaN,NaN,NaN
3,2013-01-01,25,HOME AND KITCHEN II,0.000000,NaN,NaN,NaN,NaN
4,2013-01-01,25,HOME AND KITCHEN I,0.000000,NaN,NaN,NaN,NaN
5,2013-01-01,25,HARDWARE,0.000000,NaN,NaN,NaN,NaN
6,2013-01-01,25,GROCERY II,15.000000,NaN,NaN,NaN,NaN
7,2013-01-01,25,GROCERY I,700.000000,NaN,NaN,NaN,NaN
8,2013-01-01,25,FROZEN FOODS,29.654999,NaN,NaN,NaN,NaN
9,2013-01-01,25,EGGS,46.000000,NaN,NaN,NaN,NaN


In [11]:
assert len(features_full) == len(features_lags)

for window in [7, 14, 28]:
    assert (
        f"rolling_mean_{window}"
        in features_full.columns
    )

print("✓ Filas preservadas")
print("✓ rolling_mean_7 creada")
print("✓ rolling_mean_14 creada")
print("✓ rolling_mean_28 creada")
print("✓ Día actual excluido")

features_full[
    [
        "rolling_mean_7",
        "rolling_mean_14",
        "rolling_mean_28",
    ]
].isna().mean()

✓ Filas preservadas
✓ rolling_mean_7 creada
✓ rolling_mean_14 creada
✓ rolling_mean_28 creada
✓ Día actual excluido


rolling_mean_7     0.000641
rolling_mean_14    0.000641
rolling_mean_28    0.000641
dtype: float64

In [12]:
sample = (
    features_full[
        features_full["rolling_mean_7"].notna()
    ]
    .iloc[10000]
)

mask = (
    features_base["store_nbr"].eq(
        sample["store_nbr"]
    )
    & features_base["family"].eq(
        sample["family"]
    )
    & features_base["date"].ge(
        sample["date"]
        - pd.Timedelta(days=7)
    )
    & features_base["date"].lt(
        sample["date"]
    )
)

manual_mean = features_base.loc[
    mask,
    "sales",
].mean()

print("Rolling calculada:", sample["rolling_mean_7"])
print("Rolling manual:", manual_mean)

assert np.isclose(
    sample["rolling_mean_7"],
    manual_mean,
)

print("✓ Rolling verificada sin usar el día actual")

Rolling calculada: 0.0
Rolling manual: 0.0
✓ Rolling verificada sin usar el día actual


In [13]:
series_cols = ["store_nbr", "family"]

train_positive_series = (
    train.loc[train["sales"] > 0, series_cols]
    .drop_duplicates()
)

validation_positive_series = (
    validation.loc[validation["sales"] > 0, series_cols]
    .drop_duplicates()
)

validation_new_series = (
    validation_positive_series
    .merge(
        train_positive_series,
        on=series_cols,
        how="left",
        indicator=True,
    )
)

validation_new_series = validation_new_series[
    validation_new_series["_merge"].eq("left_only")
]

print(
    "Series con primera demanda positiva "
    "recién en validación:",
    len(validation_new_series),
)

validation_new_series.head()

Series con primera demanda positiva recién en validación: 0


,store_nbr,family,_merge


In [14]:
assert len(validation_new_series) == 0

print("✓ Todas las series con demanda en validación ya eran conocidas en train")
print("✓ La definición del panel no introduce nuevas series desde el futuro")

✓ Todas las series con demanda en validación ya eran conocidas en train
✓ La definición del panel no introduce nuevas series desde el futuro


In [15]:
from src.models.baseline import (
    seasonal_weekday_baseline,
    rmsle,
)

baseline_result = seasonal_weekday_baseline(
    train=train,
    validation=validation,
    lookback_days=28,
)

baseline_result[
    [
        "date",
        "store_nbr",
        "family",
        "sales",
        "pred_baseline",
    ]
].head()

,date,store_nbr,family,sales,pred_baseline
0,2017-08-01,41,PRODUCE,2037.918,1780.99475
1,2017-08-01,41,SCHOOL AND OFFICE SUPPLIES,0.000,1.50000
2,2017-08-01,41,SEAFOOD,2.224,5.00000
3,2017-08-01,42,AUTOMOTIVE,2.000,4.50000
4,2017-08-01,42,BREAD/BAKERY,643.267,427.97750


In [16]:
baseline_rmsle = rmsle(
    baseline_result["sales"],
    baseline_result["pred_baseline"],
)

baseline_mae = np.mean(
    np.abs(
        baseline_result["sales"]
        - baseline_result["pred_baseline"]
    )
)

print(f"RMSLE baseline: {baseline_rmsle:.4f}")
print(f"MAE baseline:   {baseline_mae:.4f}")

RMSLE baseline: 0.5366
MAE baseline:   85.4780


In [17]:
assert len(baseline_result) == len(validation)

assert baseline_result["pred_baseline"].notna().all()

assert (baseline_result["pred_baseline"] >= 0).all()

print("✓ Todas las filas de validación tienen predicción")
print("✓ Sin predicciones negativas")
print("✓ Baseline evaluado sobre los mismos 15 días")

✓ Todas las filas de validación tienen predicción
✓ Sin predicciones negativas
✓ Baseline evaluado sobre los mismos 15 días


In [18]:
model_results = pd.DataFrame([
    {
        "model": "Seasonal baseline",
        "RMSLE": baseline_rmsle,
        "MAE": baseline_mae,
    }
])

model_results

,model,RMSLE,MAE
0,Seasonal baseline,0.536571,85.477984


In [19]:
MODEL_FEATURES = [
    # Identidad de la serie
    "store_nbr",
    "family",

    # Calendario
    "year",
    "month",
    "day",
    "day_of_week",
    "week_of_year",
    "is_weekend",

    # Promoción
    "onpromotion",

    # Metadata de tienda
    "city",
    "state",
    "type",
    "cluster",

    # Feriados / eventos
    "is_holiday",
    "is_event",
    "is_transfer",
    "is_additional",
    "is_bridge",
    "is_work_day",
    "is_new_year_closure",
    "event_count",

    # Historia de demanda
    "lag_1",
    "lag_7",
    "lag_14",
    "lag_28",
    "rolling_mean_7",
    "rolling_mean_14",
    "rolling_mean_28",
]

CATEGORICAL_FEATURES = [
    "store_nbr",
    "family",
    "city",
    "state",
    "type",
    "cluster",
]

TARGET = "sales"

print("Cantidad de features:", len(MODEL_FEATURES))
print("Categóricas:", len(CATEGORICAL_FEATURES))

Cantidad de features: 28
Categóricas: 6


In [20]:
train_features = features_full[
    features_full["date"] < validation_start
].copy()

X_train = train_features[MODEL_FEATURES].copy()
y_train = train_features[TARGET].copy()

for col in CATEGORICAL_FEATURES:
    X_train[col] = X_train[col].astype("category")

print("X_train:", X_train.shape)
print("y_train:", y_train.shape)

X_train.head()

X_train: (2671272, 28)
y_train: (2671272,)


,store_nbr,family,year,month,day,day_of_week,week_of_year,is_weekend,onpromotion,city,...,is_work_day,is_new_year_closure,event_count,lag_1,lag_7,lag_14,lag_28,rolling_mean_7,rolling_mean_14,rolling_mean_28
0,25,LAWN AND GARDEN,2013,1,1,1,1,0,0,Salinas,...,0,1,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,25,HOME CARE,2013,1,1,1,1,0,0,Salinas,...,0,1,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,25,HOME APPLIANCES,2013,1,1,1,1,0,0,Salinas,...,0,1,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,25,HOME AND KITCHEN II,2013,1,1,1,1,0,0,Salinas,...,0,1,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,25,HOME AND KITCHEN I,2013,1,1,1,1,0,0,Salinas,...,0,1,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [21]:
assert len(X_train) == len(y_train)

assert "sales" not in X_train.columns
assert "date" not in X_train.columns
assert "tipo_cero" not in X_train.columns
assert "familia_surtida" not in X_train.columns
assert "tienda_operativa" not in X_train.columns

assert train_features["date"].max() == pd.Timestamp("2017-07-31")

print("✓ Target separado")
print("✓ Variables diagnósticas excluidas")
print("✓ Fecha cruda excluida")
print("✓ Train termina el 31/07/2017")
print("✓ Sin información real de agosto en entrenamiento")

✓ Target separado
✓ Variables diagnósticas excluidas
✓ Fecha cruda excluida
✓ Train termina el 31/07/2017
✓ Sin información real de agosto en entrenamiento


In [22]:
missing_features = (
    X_train
    .isna()
    .mean()
    .sort_values(ascending=False)
)

missing_features[
    missing_features > 0
]

lag_28             0.020534
lag_14             0.011472
lag_7              0.006941
lag_1              0.003058
rolling_mean_28    0.000647
rolling_mean_14    0.000647
rolling_mean_7     0.000647
dtype: float64

In [23]:
%pip install lightgbm==4.7.0

Note: you may need to restart the kernel to use updated packages.


In [24]:
import lightgbm as lgb

print("LightGBM:", lgb.__version__)

LightGBM: 4.7.0


In [25]:
y_train_log = np.log1p(y_train)

lgbm_model = lgb.LGBMRegressor(
    objective="regression",

    n_estimators=500,
    learning_rate=0.05,
    num_leaves=63,

    min_child_samples=100,

    colsample_bytree=0.9,

    reg_alpha=0.1,
    reg_lambda=0.1,

    random_state=42,
    n_jobs=-1,
    verbosity=-1,
)

lgbm_model.fit(
    X_train,
    y_train_log,
    categorical_feature=CATEGORICAL_FEATURES,
)

print("✓ LightGBM entrenado")

✓ LightGBM entrenado


In [26]:
print("Árboles:", lgbm_model.booster_.num_trees())
print("Features:", lgbm_model.booster_.num_feature())

assert lgbm_model.booster_.num_feature() == len(MODEL_FEATURES)

print("✓ Modelo listo para forecasting")

Árboles: 500
Features: 28
✓ Modelo listo para forecasting


In [27]:
from src.models.recursive_forecast import recursive_forecast

recursive_predictions = recursive_forecast(
    model=lgbm_model,
    features_base=features_base,
    train=train,
    validation_start=validation_start,
    validation_end=validation_end,
    model_features=MODEL_FEATURES,
    categorical_features=CATEGORICAL_FEATURES,
)

recursive_predictions.head()

,date,store_nbr,family,prediction
0,2017-08-01,41,PRODUCE,2112.608363
1,2017-08-01,41,SCHOOL AND OFFICE SUPPLIES,0.479058
2,2017-08-01,41,SEAFOOD,6.482820
3,2017-08-01,42,AUTOMOTIVE,3.661890
4,2017-08-01,42,BREAD/BAKERY,498.183071


In [28]:
print(
    "Predicciones:",
    len(recursive_predictions),
)

print(
    "Días:",
    recursive_predictions["date"].nunique(),
)

print(
    "Desde:",
    recursive_predictions["date"].min().date(),
)

print(
    "Hasta:",
    recursive_predictions["date"].max().date(),
)

Predicciones: 25935
Días: 15
Desde: 2017-08-01
Hasta: 2017-08-15


In [29]:
validation_truth = validation[
    ["date", "store_nbr", "family", "sales"]
].copy()

lgbm_eval = validation_truth.merge(
    recursive_predictions,
    on=["date", "store_nbr", "family"],
    how="left",
    validate="one_to_one",
)

assert len(lgbm_eval) == len(validation)
assert lgbm_eval["prediction"].notna().all()
assert np.isfinite(lgbm_eval["prediction"]).all()
assert (lgbm_eval["prediction"] >= 0).all()

print("✓ Todas las predicciones emparejadas")
print("✓ Sin NaN ni infinitos")
print("✓ Sin predicciones negativas")

✓ Todas las predicciones emparejadas
✓ Sin NaN ni infinitos
✓ Sin predicciones negativas


In [30]:
lgbm_rmsle = rmsle(
    lgbm_eval["sales"],
    lgbm_eval["prediction"],
)

lgbm_mae = np.mean(
    np.abs(
        lgbm_eval["sales"]
        - lgbm_eval["prediction"]
    )
)

print(f"RMSLE LightGBM: {lgbm_rmsle:.4f}")
print(f"MAE LightGBM:   {lgbm_mae:.4f}")

RMSLE LightGBM: 0.4059
MAE LightGBM:   70.2984


In [31]:
model_results = pd.DataFrame([
    {
        "model": "Seasonal baseline",
        "RMSLE": baseline_rmsle,
        "MAE": baseline_mae,
    },
    {
        "model": "LightGBM recursive",
        "RMSLE": lgbm_rmsle,
        "MAE": lgbm_mae,
    },
])

model_results["RMSLE_vs_baseline_pct"] = (
    (baseline_rmsle - model_results["RMSLE"])
    / baseline_rmsle
    * 100
)

model_results

,model,RMSLE,MAE,RMSLE_vs_baseline_pct
0,Seasonal baseline,0.536571,85.477984,0.000000
1,LightGBM recursive,0.405895,70.298360,24.353902
